# PyTorch perf bench: SDPA backends + torch.compile

Small notebook to measure per-step time of the PixelViT in PyTorch under four configs:

| Config | Attention backend | `torch.compile` |
|---|---|---|
| `math` | `MATH` (matches existing torch baseline) | no |
| `auto` | first of `[FLASH, CUDNN, EFFICIENT, MATH]` that runs | no |
| `math + compile` | `MATH` | yes |
| `auto + compile` | first available | yes |

**Note on Flash + fp32 on T4:** FlashAttention generally requires fp16/bf16 on Turing (sm_75).
For fp32 the backend selector usually falls back to `EFFICIENT_ATTENTION` (xformers-style) or `MATH`.
The notebook reports which backend was actually picked.

Kaggle setup: 1× T4 is enough, Internet ON (to clone repo + apt is not needed).

## 1. Setup

In [ ]:
!rm -rf /tmp/hpc && git clone https://github.com/SadreevAmir/hpc_final_project /tmp/hpc && cp -r /tmp/hpc/. .
!nvidia-smi --query-gpu=index,name,memory.total --format=csv

In [ ]:
import os, glob, time, statistics
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.attention import SDPBackend, sdpa_kernel

print('torch:', torch.__version__)
print('cuda :', torch.cuda.is_available(), torch.cuda.get_device_name(0))
assert torch.cuda.is_available()
device = torch.device('cuda')
torch.manual_seed(42)

## 2. Load MNIST CSV (same logic as the other notebooks)

In [ ]:
known = [
    '/kaggle/input/digit-recognizer/train.csv',
    '/kaggle/input/fashionmnist/fashion-mnist_train.csv',
]
globbed = sorted(set(
    glob.glob('/kaggle/input/**/*train*.csv', recursive=True) +
    glob.glob('/kaggle/input/**/*Train*.csv', recursive=True)))

def valid_csv(p):
    try:
        with open(p) as f:
            return len(f.readline().split(',')) == 785
    except Exception:
        return False

CSV = next((c for c in known + globbed if os.path.exists(c) and valid_csv(c)), None)
if CSV is None:
    from torchvision.datasets import MNIST
    ds = MNIST(root='/kaggle/working/mnist_raw', train=True, download=True)
    labels = ds.targets.numpy().astype(np.int32)
    pixels = ds.data.numpy().reshape(-1, 784).astype(np.int32)
    arr = np.concatenate([labels[:, None], pixels], axis=1)
    header = 'label,' + ','.join(f'pixel{i}' for i in range(784))
    CSV = '/kaggle/working/train.csv'
    np.savetxt(CSV, arr, fmt='%d', delimiter=',', header=header, comments='')
print('CSV:', CSV)

In [ ]:
# Load all into one big GPU tensor so the bench measures compute only,
# not the dataloader.
raw = np.loadtxt(CSV, delimiter=',', skiprows=1, dtype=np.int64)
labels_all = torch.from_numpy(raw[:, 0]).to(device)
pixels_all = torch.from_numpy(raw[:, 1:]).to(device)
print('pixels:', pixels_all.shape, pixels_all.dtype)
print('labels:', labels_all.shape, labels_all.dtype)

## 3. Model (PixelViT, identical math to train_vit_torch.py — no hardcoded SDPA backend)

In [ ]:
VOCAB, SEQ, LAYERS, DIM, HEADS, CLASSES = 256, 28*28, 2, 64, 4, 10
BATCH = 8

class TransformerBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.norm1 = nn.LayerNorm(DIM, eps=1e-5)
        self.attn = nn.MultiheadAttention(DIM, HEADS, dropout=0.0, batch_first=True)
        self.norm2 = nn.LayerNorm(DIM, eps=1e-5)
        self.mlp = nn.Sequential(
            nn.Linear(DIM, 4*DIM),
            nn.GELU(approximate='tanh'),
            nn.Linear(4*DIM, DIM),
        )

    def forward(self, x):
        h = self.norm1(x)
        attn_out, _ = self.attn(h, h, h, need_weights=False)
        x = x + attn_out
        return x + self.mlp(self.norm2(x))

class PixelViT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(VOCAB, DIM)
        self.position_embedding = nn.Parameter(torch.empty(1, SEQ, DIM))
        self.blocks = nn.ModuleList(TransformerBlock() for _ in range(LAYERS))
        self.norm = nn.LayerNorm(DIM, eps=1e-5)
        self.head = nn.Linear(DIM, CLASSES, bias=False)
        self.apply(self._init)
        nn.init.normal_(self.position_embedding, mean=0, std=0.02)

    @staticmethod
    def _init(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0, std=0.02)
            if getattr(m, 'bias', None) is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.LayerNorm):
            nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
        elif isinstance(m, nn.MultiheadAttention):
            nn.init.normal_(m.in_proj_weight, mean=0, std=0.02)
            if m.in_proj_bias is not None:
                nn.init.zeros_(m.in_proj_bias)

    def forward(self, pixels):
        x = self.token_embedding(pixels) + self.position_embedding
        for blk in self.blocks:
            x = blk(x)
        return self.head(self.norm(x).mean(dim=1))

params = sum(p.numel() for p in PixelViT().parameters())
print(f'params: {params}')

## 4. Probe: which SDPA backends actually run in fp32 on this GPU?

In [ ]:
q = torch.randn(2, 4, 64, 16, device=device, dtype=torch.float32)
k = torch.randn_like(q); v = torch.randn_like(q)

backend_status = {}
for be_name in ('FLASH_ATTENTION', 'CUDNN_ATTENTION', 'EFFICIENT_ATTENTION', 'MATH'):
    if not hasattr(SDPBackend, be_name):
        backend_status[be_name] = 'enum missing in this torch'
        continue
    be = getattr(SDPBackend, be_name)
    try:
        with sdpa_kernel([be]):
            out = F.scaled_dot_product_attention(q, k, v)
            torch.cuda.synchronize()
        backend_status[be_name] = 'OK'
    except Exception as e:
        backend_status[be_name] = f'FAIL: {type(e).__name__}: {str(e)[:80]}'
for k_, v_ in backend_status.items():
    print(f'  {k_:22s} {v_}')

# Build the 'auto' priority list of backends that actually work, fp32.
AUTO_BACKENDS = [getattr(SDPBackend, n) for n in
                 ('FLASH_ATTENTION', 'CUDNN_ATTENTION', 'EFFICIENT_ATTENTION', 'MATH')
                 if hasattr(SDPBackend, n) and backend_status.get(n) == 'OK']
print('\nauto priority list:', [b.name for b in AUTO_BACKENDS])

## 5. Bench helper

In [ ]:
N = pixels_all.shape[0]

def sample_batch(B=BATCH):
    idx = torch.randint(0, N, (B,), device=device)
    return pixels_all[idx], labels_all[idx]

def bench(name, backends, use_compile, n_warmup=20, n_steps=100):
    torch.manual_seed(42)
    model = PixelViT().to(device)
    if use_compile:
        # mode='reduce-overhead' uses CUDA graphs; great fit for small models
        # with static shapes (B fixed, T fixed).
        model = torch.compile(model, mode='reduce-overhead', fullgraph=True)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)

    with sdpa_kernel(backends):
        # warmup (compile happens during these steps if use_compile=True)
        for _ in range(n_warmup):
            px, lb = sample_batch()
            opt.zero_grad(set_to_none=True)
            loss = F.cross_entropy(model(px), lb)
            loss.backward()
            opt.step()
        torch.cuda.synchronize()

        # measured loop
        ev_starts = [torch.cuda.Event(enable_timing=True) for _ in range(n_steps)]
        ev_ends   = [torch.cuda.Event(enable_timing=True) for _ in range(n_steps)]
        for i in range(n_steps):
            px, lb = sample_batch()
            ev_starts[i].record()
            opt.zero_grad(set_to_none=True)
            loss = F.cross_entropy(model(px), lb)
            loss.backward()
            opt.step()
            ev_ends[i].record()
        torch.cuda.synchronize()

    times_ms = [s.elapsed_time(e) for s, e in zip(ev_starts, ev_ends)]
    return {
        'name': name,
        'median_ms': statistics.median(times_ms),
        'p10_ms':    statistics.quantiles(times_ms, n=10)[0],
        'p90_ms':    statistics.quantiles(times_ms, n=10)[-1],
        'tput_img_s': BATCH * 1000.0 / statistics.median(times_ms),
    }

## 6. Run four configs

In [ ]:
MATH_ONLY = [SDPBackend.MATH]

configs = [
    ('math',            MATH_ONLY,     False),
    ('auto',            AUTO_BACKENDS, False),
    ('math + compile',  MATH_ONLY,     True),
    ('auto + compile',  AUTO_BACKENDS, True),
]

results = []
for name, backends, comp in configs:
    print(f'>>> {name} (backends={[b.name for b in backends]}, compile={comp})')
    r = bench(name, backends, comp)
    print(f'    median {r["median_ms"]:.3f} ms  (p10 {r["p10_ms"]:.3f}, p90 {r["p90_ms"]:.3f})'
          f'  tput {r["tput_img_s"]:.0f} img/s')
    results.append(r)

## 7. Comparison

In [ ]:
import pandas as pd
df = pd.DataFrame(results)
df['speedup_vs_math'] = df['median_ms'].iloc[0] / df['median_ms']
df.set_index('name').round(3)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 3.5))
names = [r['name'] for r in results]
med  = [r['median_ms'] for r in results]
ax.bar(names, med, color=['#888', '#377eb8', '#4daf4a', '#984ea3'])
ax.set_ylabel('median ms / step')
ax.set_title(f'PyTorch PixelViT, B={BATCH}, fp32, T4')
for i, v in enumerate(med):
    ax.text(i, v + 0.05, f'{v:.2f}', ha='center', fontsize=9)
plt.xticks(rotation=10); plt.tight_layout()
plt.savefig('torch_compile_bench.png', dpi=120)
plt.show()

## 8. (Optional) Comparison to the hand-written CUDA `all_fast`

If you've already run `kaggle_perf_bench.ipynb` and have `log_all_fast.csv`, this
cell pulls its median step time so you can compare on the same table.

In [ ]:
import csv, statistics
cuda_path = 'log_all_fast.csv'
if os.path.exists(cuda_path):
    rows = []
    with open(cuda_path) as f:
        for r in csv.DictReader(f):
            rows.append(sum(float(r[c]) for c in
                            ['t_h2d_ms','t_zero_ms','t_fwd_ms','t_bwd_ms','t_nccl_ms','t_adam_ms']))
    rows = rows[len(rows)//5:]  # discard warmup
    print(f'CUDA all_fast median step: {statistics.median(rows):.3f} ms')
    print(f'fastest PyTorch:           {df["median_ms"].min():.3f} ms ({df.loc[df["median_ms"].idxmin(), "name"]})')
else:
    print(f'no {cuda_path} found — run kaggle_perf_bench.ipynb first if you want this comparison')